In [1]:
import os
import cv2
import numpy as np
import mediapipe as mp
from tqdm import tqdm
import glob

In [ ]:
# Initialize MediaPipe Face Mesh
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(
    static_image_mode=False,
    max_num_faces=1,
    refine_landmarks=False,
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# Camera parameters 
FOCAL_LENGTH = 700 
BASELINE = 120      

# Desired frame rate
DESIRED_FPS = 60  # frames per second

def compute_3d_point(pt_left, pt_right, focal_length, baseline):
    """
    Compute the 3D coordinates of a point given its 2D coordinates in left and right images.
    """
    # Avoid division by zero
    disparity = pt_left[0] - pt_right[0]
    if disparity == 0:
        disparity = 0.1  # Minimal disparity to prevent division by zero

    depth = (focal_length * baseline) / disparity  # Z coordinate
    x = (pt_left[0] * depth) / focal_length       # X coordinate
    y = (pt_left[1] * depth) / focal_length       # Y coordinate
    return np.array([x, y, depth])

def get_facial_landmarks(image):
    """
    Detect facial landmarks in an image using MediaPipe's FaceMesh.
    Returns a list of (x, y) tuples or None if no face is detected.
    """
    results = face_mesh.process(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
    if results.multi_face_landmarks:
        # Assuming the first face detected
        landmarks = results.multi_face_landmarks[0].landmark
        h, w, _ = image.shape
        landmark_points = []
        for lm in landmarks:
            x, y = int(lm.x * w), int(lm.y * h)
            landmark_points.append((x, y))
        return landmark_points
    else:
        return None

def process_video_pair(zedl_path, zedr_path, output_path):
    """
    Process a pair of left and right videos to compute 3D facial landmarks.
    Saves the results to the specified output path.
    """
    print(f'Processing:\nLeft Video: {zedl_path}\nRight Video: {zedr_path}')

    # Open both videos
    cap_left = cv2.VideoCapture(zedl_path)
    cap_right = cv2.VideoCapture(zedr_path)

    if not cap_left.isOpened() or not cap_right.isOpened():
        print(f'Error opening videos:\nLeft: {zedl_path}\nRight: {zedr_path}')
        return

    # Get frame rates
    fps_left = cap_left.get(cv2.CAP_PROP_FPS)
    fps_right = cap_right.get(cv2.CAP_PROP_FPS)

    if fps_left == 0 or fps_right == 0:
        print(f'Error retrieving FPS for videos:\nLeft: {zedl_path}\nRight: {zedr_path}')
        cap_left.release()
        cap_right.release()
        return

    # Check if both videos have similar FPS
    if abs(fps_left - fps_right) > 1.0:
        print(f'Warning: Different FPS detected.\nLeft FPS: {fps_left}\nRight FPS: {fps_right}')
        # Optionally, you can handle resampling here
        # For simplicity, we'll proceed by taking the minimum FPS
    fps = min(fps_left, fps_right)

    # Calculate frame interval to achieve DESIRED_FPS
    frame_interval = max(int(round(fps / DESIRED_FPS)), 1)
    actual_fps = fps / frame_interval
    print(f'Original FPS: {fps:.2f}, Processing at: {actual_fps:.2f} FPS (Frame Interval: {frame_interval})')

    frame_count_left = int(cap_left.get(cv2.CAP_PROP_FRAME_COUNT))
    frame_count_right = int(cap_right.get(cv2.CAP_PROP_FRAME_COUNT))
    total_frames = min(frame_count_left, frame_count_right)

    # Calculate total frames to process based on frame interval
    frames_to_process = range(0, total_frames, frame_interval)

    # Initialize list to hold 3D landmarks for all frames
    all_3d_landmarks = []

    # Initialize frame counters
    processed_frames = 0
    num_landmarks_standard = None  # To store the standard number of landmarks

    for frame_number in tqdm(frames_to_process, desc=f'Frames in {os.path.basename(os.path.dirname(zedl_path))}'):
        # Set the position of the next frame to read
        cap_left.set(cv2.CAP_PROP_POS_FRAMES, frame_number)
        cap_right.set(cv2.CAP_PROP_POS_FRAMES, frame_number)

        ret_left, frame_left = cap_left.read()
        ret_right, frame_right = cap_right.read()

        if not ret_left or not ret_right:
            print(f'Frame read error at frame {frame_number} in videos:\nLeft: {zedl_path}\nRight: {zedr_path}')
            break

        # Detect landmarks
        landmarks_left = get_facial_landmarks(frame_left)
        landmarks_right = get_facial_landmarks(frame_right)

        if landmarks_left is None or landmarks_right is None:
            # Handle missing landmarks by appending an array filled with NaNs
            if num_landmarks_standard is None:
                # Define a standard number of landmarks, e.g., 468 for MediaPipe FaceMesh
                num_landmarks_standard = 468
            frame_3d = np.full((num_landmarks_standard, 3), np.nan)
            all_3d_landmarks.append(frame_3d)
            continue

        # Ensure consistent number of landmarks
        if num_landmarks_standard is None:
            num_landmarks_standard = len(landmarks_left)

        if len(landmarks_left) != num_landmarks_standard or len(landmarks_right) != num_landmarks_standard:
            print(f'Inconsistent number of landmarks at frame {frame_number}. Expected {num_landmarks_standard}, got {len(landmarks_left)} and {len(landmarks_right)}.')
            # Handle inconsistency by skipping or padding
            frame_3d = np.full((num_landmarks_standard, 3), np.nan)
            all_3d_landmarks.append(frame_3d)
            continue

        # Compute 3D landmarks
        frame_3d = []
        for i in range(num_landmarks_standard):
            pt_left = landmarks_left[i]
            pt_right = landmarks_right[i]
            point_3d = compute_3d_point(pt_left, pt_right, FOCAL_LENGTH, BASELINE)
            frame_3d.append(point_3d)

        all_3d_landmarks.append(np.array(frame_3d))
        processed_frames += 1

    # Release video captures
    cap_left.release()
    cap_right.release()

    print(f'Total processed frames: {processed_frames}')

    # Convert all_3d_landmarks to a NumPy array with shape (num_frames, num_landmarks, 3)
    try:
        all_3d_landmarks_array = np.stack(all_3d_landmarks)
    except ValueError as ve:
        print(f"Error stacking 3D landmarks: {ve}")
        print("Ensure that all frames have the same number of landmarks.")
        return

    # Save the 3D landmarks to a file
    try:
        np.save(output_path, all_3d_landmarks_array)
        print(f'3D landmarks saved to {output_path}\n')
    except Exception as e:
        print(f'Error saving {output_path}: {e}')

def process_videos_with_glob(parent_dir):
    """
    Traverse through all subdirectories of the parent directory to find
    pairs of L3D and R3D files using glob, and process them.
    """
    # Use glob to find all files containing L3D.mp4 recursively
    pattern_left = os.path.join(parent_dir, '**', '*L3D*.mp4')
    pattern_right = os.path.join(parent_dir, '**', '*R3D*.mp4')
    
    # Find all L3D files (left videos)
    zedl_paths = glob.glob(pattern_left, recursive=True)
    # Find all R3D files (right videos)
    zedr_paths = glob.glob(pattern_right, recursive=True)

    if not zedl_paths:
        print('No L3D.mp4 files found.')
        return
    if not zedr_paths:
        print('No R3D.mp4 files found.')
        return

    # Process each pair of left and right videos
    for zedl_path in tqdm(zedl_paths, desc="Processing L3D.mp4 files"):
        folder_path = os.path.dirname(zedl_path)
        # Look for a corresponding R3D file in the same folder
        zedr_path = os.path.join(folder_path, os.path.basename(zedl_path).replace("L3D", "R3D"))

        # Check if corresponding R3D file exists
        if zedr_path in zedr_paths:
            output_path = os.path.join(folder_path, '3D_facial_landmarks.npy')
            process_video_pair(zedl_path, zedr_path, output_path)
        else:
            print(f'Corresponding R3D.mp4 not found for {zedl_path}. Skipping this pair.\n')

if __name__ == "__main__":
    # Specify the parent directory containing subfolders
    parent_directory = '../../Video_Data' 

    if not os.path.isdir(parent_directory):
        print(f'The specified parent directory does not exist: {parent_directory}')
    else:
        process_videos_with_glob(parent_directory)

    # Close MediaPipe resources
    face_mesh.close()
